## Colab Spark EDA for Supply Chain Features

This notebook runs on a Colab T4 runtime but uses **local Spark** (`local[*]`) for EDA. It does not connect to the remote Spark cluster.

Data flow:

1. Your laptop exposes HttpFS/WebHDFS through ngrok.
2. Colab downloads the refreshed per-commodity feature parquet from HDFS path `/supply-chain/features/baseline/{commodity}` into `/content/data/features_baseline/`.
3. Local Spark reads those local parquet files and performs all EDA aggregations.
4. Plotting converts only small Spark aggregate results to pandas.

The current expected feature outputs are the six commodities: `brent`, `wti`, `copper`, `gold`, `wheat`, and `soybeans`. The latest refresh covers trading dates from April 2023 into late April 2026.

Before running this notebook, start the local WebHDFS tunnel on your laptop:

```bash
./scripts/start_hdfs_tunnel.sh
```

Paste the printed `https://...ngrok-free.app` URL into `NGROK_URL` below.

In [ ]:
# Colab setup: local Spark only. No remote Spark cluster connection.
!pip install -q pyspark requests matplotlib seaborn pandas pyarrow
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

!java -version

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path
from urllib.parse import quote

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns
from pyspark.sql import SparkSession, functions as F

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

spark = (
    SparkSession.builder
    .appName("supply-chain-eda")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

In [ ]:
# Paste your ngrok HttpFS/WebHDFS base URL here. No trailing slash.
NGROK_URL = "https://YOUR-HTTPFS.ngrok-free.app"
HDFS_USER = "root"

COMMODITIES = ["brent", "wti", "copper", "gold", "wheat", "soybeans"]
HDFS_BASELINE_BASE = "/supply-chain/features/baseline"
LOCAL_DATA_BASE = Path("/content/data/features_baseline")
FORCE_DOWNLOAD = True  # Keep True after a feature refresh so stale Colab parquet files are removed.

if "YOUR-HTTPFS" in NGROK_URL or not NGROK_URL.startswith("https://"):
    raise ValueError("Set NGROK_URL to the https://... ngrok URL for HttpFS before continuing.")

print("Reading HDFS feature base:", HDFS_BASELINE_BASE)
print("Local download base:", LOCAL_DATA_BASE)

In [ ]:
HEADERS = {"ngrok-skip-browser-warning": "true"}


def webhdfs_url(hdfs_path: str, op: str, **params) -> str:
    clean_base = NGROK_URL.rstrip("/")
    encoded_path = quote(hdfs_path, safe="/")
    query = {"op": op, "user.name": HDFS_USER, **params}
    query_string = "&".join(f"{quote(str(k))}={quote(str(v))}" for k, v in query.items())
    return f"{clean_base}/webhdfs/v1{encoded_path}?{query_string}"


def request_json(url: str) -> dict:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=60)
        resp.raise_for_status()
        return resp.json()
    except Exception as exc:
        raise RuntimeError(f"WebHDFS request failed for URL: {url}\n{exc}") from exc


def list_status(hdfs_path: str) -> list[dict]:
    url = webhdfs_url(hdfs_path, "LISTSTATUS")
    payload = request_json(url)
    return payload.get("FileStatuses", {}).get("FileStatus", [])


def download_file(hdfs_path: str, local_path: Path) -> int:
    url = webhdfs_url(hdfs_path, "OPEN")
    try:
        with requests.get(url, headers=HEADERS, timeout=120, stream=True) as resp:
            resp.raise_for_status()
            local_path.parent.mkdir(parents=True, exist_ok=True)
            size = 0
            with local_path.open("wb") as fh:
                for chunk in resp.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        fh.write(chunk)
                        size += len(chunk)
            return size
    except Exception as exc:
        raise RuntimeError(f"WebHDFS download failed for URL: {url}\n{exc}") from exc


def download_summary(commodity: str, local_dir: Path) -> dict | None:
    summary_hdfs = f"{HDFS_BASELINE_BASE}/{commodity}/_summary.json"
    url = webhdfs_url(summary_hdfs, "OPEN")
    resp = requests.get(url, headers=HEADERS, timeout=60)
    if resp.status_code == 404:
        print(f"  summary: not found at {summary_hdfs}")
        return None
    try:
        resp.raise_for_status()
    except Exception as exc:
        raise RuntimeError(f"WebHDFS summary download failed for URL: {url}\n{exc}") from exc
    out = local_dir / "_summary.json"
    out.write_bytes(resp.content)
    print(f"  summary: {out} ({out.stat().st_size:,} bytes)")
    return json.loads(resp.text)


summary_rows = []

for commodity in COMMODITIES:
    hdfs_dir = f"{HDFS_BASELINE_BASE}/{commodity}"
    local_dir = LOCAL_DATA_BASE / commodity

    if FORCE_DOWNLOAD and local_dir.exists():
        for old_file in local_dir.glob("*"):
            if old_file.is_file():
                old_file.unlink()
    local_dir.mkdir(parents=True, exist_ok=True)

    statuses = list_status(hdfs_dir)
    part_files = sorted(
        [s for s in statuses if s["type"] == "FILE" and s["pathSuffix"].endswith(".parquet")],
        key=lambda s: s["pathSuffix"],
    )
    if not part_files:
        raise RuntimeError(f"No parquet part files found in {hdfs_dir}")

    print(f"Downloading {commodity} from {hdfs_dir}")
    for status in part_files:
        suffix = status["pathSuffix"]
        size = download_file(f"{hdfs_dir}/{suffix}", local_dir / suffix)
        print(f"  {suffix}: {size:,} bytes")

    summary = download_summary(commodity, local_dir)
    if summary:
        summary_rows.append(summary)

summary_pdf = pd.DataFrame(summary_rows).sort_values("commodity").reset_index(drop=True)
print("Done downloading parquet inputs to", LOCAL_DATA_BASE)
display(summary_pdf[["commodity", "row_count", "date_min", "date_max", "class_balance", "gdelt_feature_column_count", "feature_column_count"]])

In [ ]:
dfs = {c: spark.read.parquet(str(LOCAL_DATA_BASE / c)) for c in COMMODITIES}

for commodity, df in dfs.items():
    df.cache()
    print(f"{commodity}: {df.count():,} rows, {len(df.columns):,} columns")

## Sanity Checks

All checks below run in Spark. They collect only one-row or grouped summaries.

In [ ]:
sanity_rows = []
null_count_rows = []

for commodity, df in dfs.items():
    bounds = df.agg(F.count("*").alias("rows"), F.min("event_date").alias("date_min"), F.max("event_date").alias("date_max"), F.avg("label").alias("positive_fraction")).first()
    sanity_rows.append({
        "commodity": commodity,
        "rows": bounds["rows"],
        "date_min": bounds["date_min"],
        "date_max": bounds["date_max"],
        "positive_fraction": bounds["positive_fraction"],
    })

    null_counts = df.select([
        F.count(F.when(F.col(col_name).isNull(), col_name)).alias(col_name)
        for col_name in df.columns
    ]).first().asDict()
    worst = sorted(null_counts.items(), key=lambda item: item[1], reverse=True)[:10]
    for col_name, nulls in worst:
        null_count_rows.append({"commodity": commodity, "column": col_name, "nulls": nulls})

sanity_pdf = pd.DataFrame(sanity_rows)
nulls_pdf = pd.DataFrame(null_count_rows)

display(sanity_pdf)
display(nulls_pdf)

## Plot 1: Class Balance

In [ ]:
balance_rows = []
for commodity, df in dfs.items():
    grouped = df.groupBy("label").count().toPandas()
    grouped["commodity"] = commodity
    grouped["fraction"] = grouped["count"] / grouped["count"].sum()
    balance_rows.append(grouped)

balance_pdf = pd.concat(balance_rows, ignore_index=True)

plt.figure(figsize=(10, 5))
sns.barplot(data=balance_pdf, x="commodity", y="fraction", hue="label")
plt.title("Class Balance by Commodity")
plt.ylabel("Fraction")
plt.xlabel("Commodity")
plt.ylim(0, 1)
plt.show()

## Plot 2: Label Timeline

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=True, sharey=True)
axes = axes.flatten()

for ax, commodity in zip(axes, COMMODITIES):
    pdf = dfs[commodity].select("event_date", "label").orderBy("event_date").toPandas()
    ax.scatter(pdf["event_date"], pdf["label"], s=12, alpha=0.65)
    ax.set_title(commodity)
    ax.set_yticks([0, 1])
    ax.set_ylabel("label")

plt.suptitle("High-Magnitude Label Timeline")
plt.tight_layout()
plt.show()

## Plot 3: Macro Feature Trajectories

In [ ]:
macro_pdf = (
    dfs["brent"]
    .select("event_date", "treasury_10y", "usd_index")
    .orderBy("event_date")
    .toPandas()
)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(macro_pdf["event_date"], macro_pdf["treasury_10y"], label="treasury_10y", color="tab:blue")
ax1.set_ylabel("treasury_10y", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(macro_pdf["event_date"], macro_pdf["usd_index"], label="usd_index", color="tab:orange")
ax2.set_ylabel("usd_index", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

plt.title("Macro Feature Trajectories")
fig.tight_layout()
plt.show()

## Plot 4: Volatility by Commodity

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=True)
axes = axes.flatten()

for ax, commodity in zip(axes, COMMODITIES):
    pdf = dfs[commodity].select("event_date", "volatility_20d").orderBy("event_date").toPandas()
    ax.plot(pdf["event_date"], pdf["volatility_20d"])
    ax.set_title(commodity)
    ax.set_ylabel("volatility_20d")

plt.suptitle("20-Day Volatility")
plt.tight_layout()
plt.show()

## Plot 5: Chokepoint Event Totals

In [ ]:
chokepoints = ["hormuz", "suez", "red_sea", "black_sea", "malacca", "panama", "taiwan", "chile"]

event_total_exprs = [F.sum(F.col(f"{cp}_event_count_1d")).alias(cp) for cp in chokepoints]
event_totals = dfs["brent"].agg(*event_total_exprs).first().asDict()
event_totals_pdf = pd.DataFrame(
    [{"chokepoint": cp, "event_count_1d_total": event_totals[cp]} for cp in chokepoints]
).sort_values("event_count_1d_total", ascending=False)

plt.figure(figsize=(11, 5))
sns.barplot(data=event_totals_pdf, x="chokepoint", y="event_count_1d_total")
plt.xticks(rotation=30, ha="right")
plt.title("Chokepoint Event Totals from 1-Day Counts")
plt.ylabel("Total event_count_1d")
plt.xlabel("Chokepoint")
plt.tight_layout()
plt.show()

## Plot 6: Brent GDELT Event Count vs Label Correlation

In [ ]:
windows = ["1d", "3d", "7d", "30d"]
correlation_rows = []
brent = dfs["brent"]

for cp in chokepoints:
    row = {"chokepoint": cp}
    for window in windows:
        col_name = f"{cp}_event_count_{window}"
        row[window] = brent.stat.corr(col_name, "label")
    correlation_rows.append(row)

corr_pdf = pd.DataFrame(correlation_rows).set_index("chokepoint")

plt.figure(figsize=(8, 6))
sns.heatmap(corr_pdf, annot=True, cmap="vlag", center=0, fmt=".3f")
plt.title("Brent: event_count Window Correlation with Label")
plt.xlabel("Window")
plt.ylabel("Chokepoint")
plt.tight_layout()
plt.show()

## Plot 7: Cross-Commodity Label Correlation

In [ ]:
label_wide = None
for commodity, df in dfs.items():
    labels = df.select("event_date", F.col("label").cast("double").alias(commodity))
    label_wide = labels if label_wide is None else label_wide.join(labels, on="event_date", how="inner")

label_pdf = label_wide.orderBy("event_date").toPandas()
label_corr = label_pdf[COMMODITIES].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(label_corr, annot=True, cmap="vlag", center=0, vmin=-1, vmax=1, fmt=".2f")
plt.title("Cross-Commodity Label Correlation")
plt.tight_layout()
plt.show()

## Plot 8: Feature Missingness Heatmap

In [ ]:
missing_rows = []
for commodity, df in dfs.items():
    feature_cols = [c for c in df.columns if c not in ("event_date", "label", "split")]
    row_count = df.count()
    null_counts = df.select([
        F.count(F.when(F.col(col_name).isNull(), col_name)).alias(col_name)
        for col_name in feature_cols
    ]).first().asDict()

    for col_name, nulls in null_counts.items():
        missing_rows.append({
            "commodity": commodity,
            "feature": col_name,
            "missing_fraction": nulls / row_count if row_count else 0,
        })

missing_pdf = pd.DataFrame(missing_rows)
# Show the 40 most-missing features across commodities to keep the heatmap readable.
top_features = (
    missing_pdf.groupby("feature")["missing_fraction"]
    .mean()
    .sort_values(ascending=False)
    .head(40)
    .index
)
missing_matrix = (
    missing_pdf[missing_pdf["feature"].isin(top_features)]
    .pivot(index="feature", columns="commodity", values="missing_fraction")
    .loc[top_features]
)

plt.figure(figsize=(8, 12))
sns.heatmap(missing_matrix, cmap="mako", vmin=0, vmax=1)
plt.title("Feature Missingness: Top 40 Features")
plt.xlabel("Commodity")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()